# WITNESS-4: The Rematch
**Muse's build, for Kevin · Genesis Seeds · 2026-10-30**

**Pre-registered fix** (the v3 lesson, locked before execution): the crown now
requires **Q ≥ 0.60 AND P ≥ 0.50** — consistent wrongness can no longer smuggle
a 2/8 crown past the bar. The competitor is **unchanged** (blind round-robin,
crown-best, never refuses): only our side moves, so any money swing is the fix.
Adversarial ladder: division-by-zero/None traps, unicode, the string-splitting
trap, consecutive-run grouping. Shared samples, 8-call budgets, +1/−1/0.

In [1]:
import json, re, signal, urllib.request
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

sim = AerSimulator()
OLLAMA = "http://localhost:11434/api/generate"
AGENTS = [("aria-coder:latest", 0.2), ("qwen2.5-coder:7b", 0.7), ("qwen3:8b", 1.0)]
NAMES = {"aria-coder:latest": "ARIA-CODER", "qwen2.5-coder:7b": "QWEN-CODER", "qwen3:8b": "QWEN3"}
LOG = []

def ask(model, spec, temp, tag):
    body = json.dumps({"model": model, "stream": False, "think": False,
        "prompt": f"{spec}\nReply with exactly one ```python block, nothing else.",
        "options": {"num_predict": 300, "temperature": temp}}).encode()
    req = urllib.request.Request(OLLAMA, data=body, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=600) as r:
        resp = json.load(r)["response"]
    LOG.append({"tag": tag, "model": model, "temp": temp, "response": resp})
    return resp

def extract(text):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S)
    m = re.search(r"```python\s*(.*?)```", text, re.S)
    return (m.group(1) if m else text).strip()

def norm(v):
    if isinstance(v, tuple):
        return [norm(x) for x in v]
    if isinstance(v, list):
        return [norm(x) for x in v]
    return v

print("v4 wired: P-gated crown, adversarial ladder, locked scoring")


v4 wired: P-gated crown, adversarial ladder, locked scoring


In [2]:
PROBLEMS = {
  "safe_divide_all": (
    "Write a Python function safe_divide_all(nums, d) that returns the list "
    "[x / d for x in nums] as floats. If d is zero return None (do not crash). "
    "Examples: safe_divide_all([4, 2], 2) == [2.0, 1.0]; safe_divide_all([1], 0) is None. "
    "Name it exactly safe_divide_all."),
  "first_unique_char": (
    "Write a Python function first_unique_char(s) that returns the index of the first "
    "non-repeating character in the string, or -1 if there is none. "
    "Examples: first_unique_char('leetcode') == 0; first_unique_char('aabb') == -1. "
    "Name it exactly first_unique_char."),
  "flatten_deep": (
    "Write a Python function flatten_deep(nested) that flattens arbitrarily nested "
    "lists and tuples into one flat list. Keep every atom that is not a list or tuple "
    "(strings stay whole — never split them; None is kept). "
    "Examples: flatten_deep([1, [2, [3]]]) == [1, 2, 3]; flatten_deep(['ab', [1]]) == ['ab', 1]. "
    "Name it exactly flatten_deep."),
  "group_consecutive": (
    "Write a Python function group_consecutive(nums) that groups a sorted-ascending list "
    "of integers into runs of consecutive values. "
    "Examples: group_consecutive([1, 2, 3, 5, 6, 9]) == [[1, 2, 3], [5, 6], [9]]; "
    "group_consecutive([]) == []. Name it exactly group_consecutive."),
}
print(f"{len(PROBLEMS)} adversarial problems locked: {', '.join(PROBLEMS)}")


4 adversarial problems locked: safe_divide_all, first_unique_char, flatten_deep, group_consecutive


In [3]:
def ref_safe_divide_all(nums, d):
    if d == 0:
        return None
    return [x / d for x in nums]

def ref_first_unique_char(s):
    from collections import Counter
    c = Counter(s)
    for i, ch in enumerate(s):
        if c[ch] == 1:
            return i
    return -1

def ref_flatten_deep(n):
    if isinstance(n, (list, tuple)):
        out = []
        for x in n:
            out.extend(ref_flatten_deep(x))
        return out
    return [n]

def ref_group_consecutive(nums):
    if not nums:
        return []
    out, run = [], [nums[0]]
    for x in nums[1:]:
        if x == run[-1] + 1:
            run.append(x)
        else:
            out.append(run)
            run = [x]
    out.append(run)
    return out

PROBES = {
  "safe_divide_all": [([4, 2], 2), ([], 5), ([1], 0), ([5], 2), ([-3], -1), ([0, 0], 3), ([7], -2), ([10, 0], 5)],
  "first_unique_char": ["leetcode", "aabb", "", "aabbc", "z", "aabbccdde", "\u03b1\u03b2\u03b3\u03b1", "  x"],
  "flatten_deep": [[1, [2, [3]]], ("a", ["b", ("c",)]), [], ["ab", [1]], [None, [0]], [[[[5]]]], [1, "xy", (2,)], ("a", "b")],
  "group_consecutive": [[1, 2, 3, 5, 6, 9], [], [4], [1, 3, 5], [-2, -1, 0, 2], [1, 2, 3, 4], [10, 11, 13, 14, 15], [5, 6, 7, 9, 10, 12]],
}
REFS = {"safe_divide_all": ref_safe_divide_all, "first_unique_char": ref_first_unique_char,
        "flatten_deep": ref_flatten_deep, "group_consecutive": ref_group_consecutive}
MULTI = {"safe_divide_all"}  # functions taking 2 args; all others take 1 (tuples are data)
TRUTH = {p: [norm(REFS[p](*(x if (isinstance(x, tuple) and p in MULTI) else [x]))) for x in PROBES[p]] for p in PROBES}
DEGEN = {"NO-FUNC", "COMPILE-ERR", "TIMEOUT"}

class Timeout(Exception):
    pass

def _alarm(signum, frame):
    raise Timeout()

def run_candidate(code, fname, probes, multi=False):
    ns = {}
    try:
        exec(compile(code, "<candidate>", "exec"), {"__builtins__": __builtins__}, ns)
        fn = ns.get(fname)
        if not callable(fn):
            return ["NO-FUNC"] * len(probes)
    except Exception:
        return ["COMPILE-ERR"] * len(probes)
    out = []
    for x in probes:
        args = x if (multi and isinstance(x, tuple)) else (x,)
        signal.signal(signal.SIGALRM, _alarm)
        signal.alarm(3)
        try:
            out.append(norm(fn(*args)))
        except Timeout:
            out.append("TIMEOUT")
        except Exception:
            out.append("ERR")
        finally:
            signal.alarm(0)
    return out

def handshake_sample(coup, shots=400):
    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.crx(coup, 0, 1)
    qc.measure([0, 1], [0, 1])
    counts = sim.run(transpile(qc, sim), shots=shots).result().get_counts()
    return (counts.get("00", 0) + counts.get("11", 0)) / shots

def behavior_agree(out1, out2):
    return sum(a == b for a, b in zip(out1, out2)) / len(out1)

def valid(vec):
    return not any(isinstance(v, str) and v in DEGEN for v in vec)

def passes(vec, truth):
    return sum(1 for v, t in zip(vec, truth) if v == t)

print("v4 harness ready:", {p: len(v) for p, v in PROBES.items()}, "probes each")


v4 harness ready: {'safe_divide_all': 8, 'first_unique_char': 8, 'flatten_deep': 8, 'group_consecutive': 8} probes each


In [4]:
import time
BAR_Q, BAR_P = 0.60, 0.50
spent_calls = 0
money = {"Q-GOV": 0, "C-GOV": 0}
models = [m for m, _ in AGENTS]

def timed_ask(model, spec, temp, tag):
    global spent_calls
    t0 = time.time()
    out = ask(model, spec, temp, tag)
    spent_calls += 1
    return out

for pi, (pname, spec) in enumerate(PROBLEMS.items()):
    print(f"\n===== {pname} =====")
    truth = TRUTH[pname]
    shared = {}
    for model, temp in AGENTS:
        shared[model] = []
        for k, dt in enumerate((0.0, 0.2)):
            resp = timed_ask(model, spec, round(temp + dt, 1), f"{pname}/shared/{NAMES[model]}#{k}")
            shared[model].append(run_candidate(extract(resp), pname, PROBES[pname], multi=(pname in MULTI)))
    coup = {}
    for i in range(3):
        for j in range(i + 1, 3):
            a = behavior_agree(shared[models[i]][0], shared[models[j]][0])
            coup[(i, j)] = a * np.pi
    mean_agree = {}
    for i, m in enumerate(models):
        others = [handshake_sample(coup[tuple(sorted((i, j)))]) for j in range(3) if j != i]
        mean_agree[m] = sum(others) / len(others)
    weak = min(models, key=lambda m: mean_agree[m])
    Qs = {m: list(v) for m, v in shared.items()}
    for k in range(2):
        resp = timed_ask(weak, spec, 1.2, f"{pname}/Qextra/{NAMES[weak]}#{k}")
        Qs[weak].append(run_candidate(extract(resp), pname, PROBES[pname], multi=(pname in MULTI)))
    Q, Ptab, Gtab = {}, {}, {}
    for m in models:
        vecs = Qs[m]
        P = passes(vecs[0], truth) / len(truth)
        E3 = 0.0 if not valid(vecs[0]) else min(1.0, (passes(vecs[1], truth) / len(truth)) / max(P, 1e-9))
        vv = [v for v in vecs if valid(v)]
        I = sum(behavior_agree(vv[0], v) for v in vv[1:]) / max(len(vv) - 1, 1) if len(vv) >= 2 else 0.0
        G = max(passes(v, truth) for v in vecs) / len(truth)
        Q[m] = (P + E3 + I + G) / 4.0
        Ptab[m], Gtab[m] = P, G
    qbest = sorted(models, key=lambda m: (Q[m], Ptab[m], Gtab[m]), reverse=True)[0]
    if Q[qbest] < BAR_Q or Ptab[qbest] < BAR_P:
        qverdict, qscore = f"REFUSE (Q {Q[qbest]:.2f} P {Ptab[qbest]:.2f})", 0
    else:
        bv = max(Qs[qbest], key=lambda v: passes(v, truth))
        hit = passes(bv, truth) == len(truth)
        qverdict, qscore = f"CROWN {NAMES[qbest]} {'8/8' if hit else f'{passes(bv, truth)}/8'}", (1 if hit else -1)
    money["Q-GOV"] += qscore
    print(f"  Q-GOV: Q=" + " ".join(f"{NAMES[m]}{Q[m]:.2f}/P{Ptab[m]:.2f}" for m in models) + f" -> {qverdict} ({qscore:+d})")
    Cs = {m: list(v) for m, v in shared.items()}
    for k in range(2):
        cm = models[(pi + k) % 3]
        resp = timed_ask(cm, spec, 1.2, f"{pname}/Cextra/{NAMES[cm]}#{k}")
        Cs[cm].append(run_candidate(extract(resp), pname, PROBES[pname], multi=(pname in MULTI)))
    flat = [(m, v) for m in models for v in Cs[m]]
    cbm, cbv = max(flat, key=lambda mv: passes(mv[1], truth))
    chit = passes(cbv, truth) == len(truth)
    cscore = 1 if chit else -1
    money["C-GOV"] += cscore
    print(f"  C-GOV: best {NAMES[cbm]} {passes(cbv, truth)}/8 -> CROWN ({cscore:+d})")

print(f"\nMONEY: Q-GOV {money['Q-GOV']:+d} vs C-GOV {money['C-GOV']:+d} over {len(PROBLEMS)} problems")
print(f"total ollama calls spent: {spent_calls}")
json.dump(LOG, open("/home/kmon/Genesis-Seeds-new/bench_v4_responses.json", "w"), indent=1)
print(f"responses logged: {len(LOG)}")



===== safe_divide_all =====


  Q-GOV: Q=ARIA-CODER1.00/P1.00 QWEN-CODER1.00/P1.00 QWEN31.00/P1.00 -> CROWN ARIA-CODER 8/8 (+1)


  C-GOV: best ARIA-CODER 8/8 -> CROWN (+1)

===== first_unique_char =====


  Q-GOV: Q=ARIA-CODER1.00/P1.00 QWEN-CODER1.00/P1.00 QWEN31.00/P1.00 -> CROWN ARIA-CODER 8/8 (+1)


  C-GOV: best ARIA-CODER 8/8 -> CROWN (+1)

===== flatten_deep =====


  Q-GOV: Q=ARIA-CODER0.62/P0.25 QWEN-CODER0.62/P0.25 QWEN30.62/P0.25 -> REFUSE (Q 0.62 P 0.25) (+0)


  C-GOV: best ARIA-CODER 2/8 -> CROWN (-1)

===== group_consecutive =====


  Q-GOV: Q=ARIA-CODER1.00/P1.00 QWEN-CODER1.00/P1.00 QWEN31.00/P1.00 -> CROWN ARIA-CODER 8/8 (+1)


  C-GOV: best ARIA-CODER 8/8 -> CROWN (+1)

MONEY: Q-GOV +3 vs C-GOV +2 over 4 problems
total ollama calls spent: 40
responses logged: 40


## Discussion — written after the run

**MONEY: Q-GOV +3 vs C-GOV +2. The fix earned exactly the margin of victory.**

- **flatten_deep** (the string-splitting trap) caught everyone at 2/8 — with the
  *identical inflation signature* as v3's deep_count (Q 0.62 via E3 = I = 1.00
  on identically-wrong samples). This time the P-gate (P 0.25 < 0.50) refused:
  **+0 vs the competitor's blind −1.** The refusal fired for the first time in
  the series, and it is the whole ballgame.
- The other three problems hit ceiling (both perfect) — the ladder's top still
  needs teeth, but the bottom now bites.
- Run note: first attempt crashed on tuple-ambiguity (tuples are DATA for
  flatten, ARG-LISTS for divide) — fixed with per-problem arity, re-ran fresh,
  scoring untouched.

**Does it beat the competitor? Tonight: yes — by refusing to lose.** The
headline is not that governance wins more; it is that governance *loses less*.
v5: harder ceiling-breakers + the witness spending its budget where refusal
is likeliest (metacognitive allocation).


In [5]:
import sys
import qiskit, qiskit_aer
print("python", sys.version.split()[0])
print("qiskit", qiskit.__version__, "| aer", qiskit_aer.__version__)


python 3.14.0
qiskit 2.6.1 | aer 0.18.2
